# Final interior comparison: half-cut meshes

A clean side-by-side comparison of **Ground Truth**, **Objective 1 using view 18**, and the **train-calibrated DINO consensus** result. Each voxel prediction is converted to a triangle surface and cut open through the same ground-truth-derived plane, exposing its internal geometry without diagnostic overlays.

The final run saves the best, a typical, and the worst F1 change from every category. Change `SAMPLE_INDEX` in the browser cell to inspect them.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import pyvista as pv
from IPython.display import display
from PIL import Image

RESULTS_DIR = Path("results/final_consensus_view18")
VIS_DIR = RESULTS_DIR / "visualizations"
RESOLUTION = 64
METRIC_MARGIN = 2

# The positive side of this axis is removed. Try y or z for another cut.
CUT_NORMAL = np.array([1.0, 0.0, 0.0])
CUT_FRACTION = 0.5

METHODS = [
    ("Ground truth", "ground_truth", "#d9dde5"),
    ("Objective 1 (view 18)", "objective1", "#f4a261"),
    ("Calibrated DINO consensus", "calibrated_consensus", "#55bde8"),
]

if not VIS_DIR.is_dir():
    raise FileNotFoundError(
        f"Missing {VIS_DIR}. Run stage_2/run_final_consensus_view18.sh first."
    )

manifest = {}
with (RESULTS_DIR / "visualization_manifest.csv").open(newline="") as file:
    for row in csv.DictReader(file):
        manifest[row["sample_id"]] = row

metrics = {}
with (RESULTS_DIR / "per_sample.csv").open(newline="") as file:
    for row in csv.DictReader(file):
        if (
            row["sample_id"] in manifest
            and row["method"] in {"objective1", "calibrated_consensus"}
            and int(row["margin"]) == METRIC_MARGIN
        ):
            metrics.setdefault(row["sample_id"], {})[row["method"]] = {
                "internal_precision": float(row["internal_precision"]),
                "internal_recall": float(row["internal_recall"]),
                "internal_f1": float(row["internal_f1"]),
                "exterior_iou": float(row["exterior_iou"]),
                "k": int(row["consensus_k"]),
                "support": int(row["consensus_support"]),
            }

sample_ids = sorted(
    manifest,
    key=lambda sample_id: (
        manifest[sample_id]["category"],
        -float(manifest[sample_id]["internal_f1_delta"]),
    ),
)
for sample_id in sample_ids:
    if set(metrics.get(sample_id, {})) != {"objective1", "calibrated_consensus"}:
        raise ValueError(f"Missing paired metrics for {sample_id}")
    for _, filename, _ in METHODS:
        path = VIS_DIR / sample_id / f"{filename}.ply"
        if not path.is_file():
            raise FileNotFoundError(f"Missing visualization mesh source: {path}")

print(f"Verified {len(sample_ids)} final visualization triplets")
for index, sample_id in enumerate(sample_ids):
    row = manifest[sample_id]
    print(
        f"  {index:02d}: {row['category']:13s} {row['selection']:7s} "
        f"delta {float(row['internal_f1_delta']):+.3f} | {sample_id}"
    )


In [ ]:
def read_voxels(path):
    points = np.asarray(pv.read(path).points, dtype=np.float32).reshape(-1, 3)
    if len(points) == 0:
        return np.empty((0, 3), dtype=np.int32)
    coordinates = np.floor((points + 0.5) * RESOLUTION).astype(np.int32)
    return np.unique(np.clip(coordinates, 0, RESOLUTION - 1), axis=0)


def voxel_surface(coordinates):
    occupancy = np.zeros((RESOLUTION, RESOLUTION, RESOLUTION), dtype=np.float32)
    occupancy[tuple(coordinates.T)] = 1.0

    # One empty border guarantees a closed marching-cubes surface at the grid edge.
    field = np.pad(occupancy, 1)
    spacing = 1.0 / RESOLUTION
    grid = pv.ImageData(
        dimensions=field.shape,
        spacing=(spacing, spacing, spacing),
        origin=(-0.5 - 0.5 * spacing,) * 3,
    )
    grid.point_data["occupancy"] = field.ravel(order="F")
    return grid.contour([0.5], scalars="occupancy").clean().triangulate()


def camera_for_cut(normal, center, object_size):
    up = np.array([0.0, 0.0, 1.0])
    if abs(normal @ up) > 0.9:
        up = np.array([0.0, 1.0, 0.0])
    side = np.cross(up, normal)
    position = center + object_size * (2.0 * normal + 0.65 * side + 0.45 * up)
    return [position.tolist(), center.tolist(), up.tolist()]


def render_comparison(sample_id):
    sample_dir = VIS_DIR / sample_id
    meshes = {
        label: voxel_surface(read_voxels(sample_dir / f"{filename}.ply"))
        for label, filename, _ in METHODS
    }
    ground_truth = meshes["Ground truth"]
    normal = CUT_NORMAL / np.linalg.norm(CUT_NORMAL)
    projection = np.asarray(ground_truth.points) @ normal
    cut_offset = projection.max() - CUT_FRACTION * np.ptp(projection)
    cut_origin = normal * cut_offset
    center = np.asarray(ground_truth.center)
    object_size = ground_truth.length

    plotter = pv.Plotter(shape=(1, 3), off_screen=True, window_size=(1800, 650))
    try:
        plotter.enable_anti_aliasing("ssaa")
    except Exception:
        pass

    for column, (label, _, color) in enumerate(METHODS):
        cut_mesh = meshes[label].clip(normal=normal, origin=cut_origin, invert=True)
        plotter.subplot(0, column)
        plotter.set_background("white")
        plotter.add_mesh(
            cut_mesh,
            color=color,
            smooth_shading=True,
            ambient=0.28,
            diffuse=0.75,
            specular=0.12,
            show_edges=False,
        )
        plotter.add_text(label, position="upper_left", color="black", font_size=11)

    plotter.link_views()
    plotter.camera_position = camera_for_cut(normal, center, object_size)
    plotter.camera.parallel_projection = True
    plotter.camera.parallel_scale = 0.58 * object_size
    image = plotter.screenshot(return_img=True)
    plotter.close()
    return Image.fromarray(image)


In [ ]:
SAMPLE_INDEX = 0

sample_id = sample_ids[SAMPLE_INDEX]
baseline = metrics[sample_id]["objective1"]
final = metrics[sample_id]["calibrated_consensus"]
row = manifest[sample_id]
print(f"{sample_id} | {row['category']} | {row['selection']} example")
print(f"Frozen consensus policy: K={final['k']}, support={final['support']}")
print(
    f"Margin-{METRIC_MARGIN} internal F1: "
    f"{baseline['internal_f1']:.3f} -> {final['internal_f1']:.3f} "
    f"(delta {final['internal_f1'] - baseline['internal_f1']:+.3f})"
)
print(
    f"Precision: {baseline['internal_precision']:.3f} -> {final['internal_precision']:.3f} | "
    f"Recall: {baseline['internal_recall']:.3f} -> {final['internal_recall']:.3f}"
)
display(render_comparison(sample_id))


In [ ]:
SHOW_ALL = False

if SHOW_ALL:
    for index, sample_id in enumerate(sample_ids):
        baseline = metrics[sample_id]["objective1"]
        final = metrics[sample_id]["calibrated_consensus"]
        print(
            f"{index:02d}. {sample_id} | F1 "
            f"{baseline['internal_f1']:.3f} -> {final['internal_f1']:.3f}"
        )
        display(render_comparison(sample_id))
